# Create region segmentation

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from natsort import natsorted
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set IO path
base_path = '/home/jiahao/wanglab/Data/Analyzed/2024-12-02-Mingrui-SCZ/'
expr_path = os.path.join(base_path, 'expr')
fig_path = os.path.join(base_path, 'figures')
output_path = os.path.join(base_path, 'output')
os.makedirs(output_path, exist_ok=True)

sc.settings.figdir = fig_path
input_path = os.path.join(expr_path, 'tissue region identification')

## Input

In [ ]:
# # PFC
# current_region = "PFC"
# cast_output_path = os.path.join(output_path, current_region)
# os.makedirs(cast_output_path, exist_ok=True)

# adata = sc.read_h5ad(os.path.join(input_path, '2025-01-13-pfc-rgn-label.h5ad'))
# adata

In [ ]:
# # ST
# current_region = "ST"
# cast_output_path = os.path.join(output_path, current_region)
# os.makedirs(cast_output_path, exist_ok=True)

# adata = sc.read_h5ad(os.path.join(input_path, '2025-01-13-st-rgn-label.h5ad'))
# adata

In [ ]:
# HP
current_region = "HP"
cast_output_path = os.path.join(output_path, current_region)
os.makedirs(cast_output_path, exist_ok=True)

adata = sc.read_h5ad(os.path.join(input_path, '2025-01-13-hp-rgn-label.h5ad'))
adata

## SPIN post processing

### SPIN smoothing

In [ ]:
scaling_factor = 0.25
adata.obs['region_codes'] = adata.obs['region_label'].cat.codes + 1
adata.obs['region_codes'] = adata.obs['region_codes'].astype(object)

adata.obs['global_x_scaled'] = (adata.obs['global_x'] * scaling_factor).astype(int)
adata.obs['global_y_scaled'] = (adata.obs['global_y'] * scaling_factor).astype(int)

In [ ]:
# kNN smoothing
for current_sample in natsorted(adata.obs['sample_id'].unique()):
    
    print(f"Processing {current_sample}")
    sample_obs = adata.obs[adata.obs['sample_id'] == current_sample].copy()
    sample_obs['region_codes'] = sample_obs['region_label'].cat.codes + 1
    current_coords = sample_obs[['global_x_scaled', 'global_y_scaled']].values

    # find the nearest neighbor for each cell
    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=50).fit(current_coords)
    distances, indices = nbrs.kneighbors(current_coords)

    for i in range(indices.shape[0]):
        current_vote = sample_obs.iloc[indices[i, 1:], :]['region_codes'].mode().values[0]
        adata.obs.loc[sample_obs.index[i], 'region_codes'] = current_vote


In [ ]:
# Visualization
for current_sample in natsorted(adata.obs['sample_id'].unique()):
    
    print(f"Plot {current_sample}")
    sample_obs = adata.obs[adata.obs['sample_id'] == current_sample].copy()
    # sample_obs = sample_obs[sample_obs['region_codes'] != 'NA']
    sample_obs['region_codes'] = sample_obs['region_codes'].astype('category')

    figsize=((int(sample_obs['global_x'].max() * scaling_factor) + 1) / 1000, (int(sample_obs['global_y'].max() * scaling_factor) + 1) / 1000)
    region_img = np.zeros((int(sample_obs['global_y'].max() * scaling_factor) + 1, int(sample_obs['global_x'].max() * scaling_factor) + 1))

    fig, ax = plt.subplots(figsize=figsize)
    plt.imshow(region_img, cmap='Greys_r')
    sns.scatterplot(x='global_x_scaled', y='global_y_scaled', data=sample_obs,
                    hue='region_codes', palette='tab10', s=3, alpha=1, legend=False, edgecolor=None) 

    plt.axis('off')
    plt.tight_layout(pad=0)
    current_output_path = os.path.join(output_path, 'spin_mask', current_sample)
    os.makedirs(current_output_path, exist_ok=True)
    plt.savefig(os.path.join(current_output_path, 'spin_smoothed.tif'), dpi=1000)

In [ ]:
adata.obs['region_codes'] = adata.obs['region_codes'].astype(str)
adata.obs['region_codes'] = adata.obs['region_codes'].astype('category')
adata.obs['region_codes'] = adata.obs['region_codes'].cat.reorder_categories(natsorted(adata.obs['region_codes'].unique()))

In [ ]:
adata.write_h5ad(os.path.join(input_path, '2025-03-24-hp-rgn-label.h5ad'))

## Single sample test

In [ ]:
current_sample = 'sample1'

current_obs = adata.obs[adata.obs['sample_id'] == current_sample].copy()
current_obs = current_obs[current_obs['region_codes'] != 'NA']

### Create bg img

In [ ]:
from skimage import img_as_ubyte
from skimage.filters import gaussian
from skimage.segmentation import expand_labels, find_boundaries
from scipy.ndimage import binary_fill_holes
from skimage.morphology import convex_hull_image, remove_small_objects
from skimage.measure import label
from skimage.color import label2rgb


scaling_factor = 0.25
img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))
img[(current_obs['global_y'] * scaling_factor).astype(int).values, (current_obs['global_x'] * scaling_factor).astype(int).values] = current_obs['region_codes'].values

img = expand_labels(img, distance=30)
bw_img = img > 0
img_uint8 = img_as_ubyte(bw_img)
img_gaussian = gaussian(img_uint8, sigma=3)
bw_img = img_gaussian > 0
bg_img = binary_fill_holes(bw_img)

In [ ]:
fig, ax = plt.subplots()
plt.imshow(bg_img)

### Create seed img

In [ ]:
from skimage.measure import regionprops
seed_img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))

for current_region in tqdm(natsorted(current_obs['region_codes'].unique())):
# for current_region in tqdm([10]):
    print(current_region)
    region_obs = current_obs.loc[current_obs['region_codes'] == current_region, :]

    region_img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))
    region_img[(region_obs['global_y'] * scaling_factor).astype(int).values, (region_obs['global_x'] * scaling_factor).astype(int).values] = 1

    # region_img = expand_labels(region_img, distance=20)
    bw_img = region_img > 0
    img_uint8 = img_as_ubyte(bw_img)
    img_gaussian = gaussian(img_uint8, sigma=2)
    bw_img = img_gaussian > 0
    bw_img = binary_fill_holes(bw_img)

    objects = label(bw_img)

    # if objects.max() > 10:
    #     props = regionprops(objects)
    #     areas = [prop.area for prop in props]
    #     threshold = np.percentile(areas, 10)
    #     fig, ax = plt.subplots()
    #     sns.histplot(areas)

    #     large_objects = remove_small_objects(objects, min_size=threshold)
    #     seed_img[large_objects > 0] = current_region

    #     current_obj_preview = label2rgb(large_objects, bg_label=0)
    #     fig, ax = plt.subplots()
    #     plt.imshow(current_obj_preview)

    # else:
        # culls = convex_hull_object(objects)
    seed_img[objects > 0] = current_region

    current_obj_preview = label2rgb(objects, bg_label=0)
    fig, ax = plt.subplots()
    plt.imshow(current_obj_preview)
    


In [ ]:
from skimage.segmentation import watershed
labels = watershed(bg_img, markers=seed_img.astype(int), mask=bg_img)

In [ ]:
fig, ax = plt.subplots()
# plt.imshow(labels)
plt.imshow(label2rgb(labels, bg_label=0))

In [ ]:
from skimage.measure import find_contours
label_bnd = find_boundaries(labels)
contours = find_contours(label_bnd, level=.8)
blank_img = np.zeros((int(current_obs['global_x'].max() * scaling_factor) + 1, int(current_obs['global_y'].max() * scaling_factor) + 1))

In [ ]:
figsize=((int(current_obs['global_x'].max() * scaling_factor) + 1) / 1000, (int(current_obs['global_y'].max() * scaling_factor) + 1) / 1000)
fig, ax = plt.subplots(figsize=figsize)

plt.imshow(blank_img, cmap='Greys')
sns.scatterplot(x='global_x_scaled', y='global_y_scaled', hue='region_codes', data=current_obs, 
                palette='tab10', s=3, alpha=1, legend=False, edgecolor=None)

# sns.scatterplot(x='global_x_scaled', y='global_y_scaled', color='b', data=current_obs, 
#                 s=3, alpha=1, legend=False, edgecolor=None)
for contour in contours:
    ax.plot(contour[:, 1], contour[:, 0], linewidth=.7, color='#5c5c5c')

plt.axis('off')
plt.tight_layout()
# plt.savefig(os.path.join(current_output_path, 'test.tif'))

## Batch

In [ ]:
from skimage import img_as_ubyte
from skimage.filters import gaussian
from skimage.segmentation import expand_labels, find_boundaries
from scipy.ndimage import binary_fill_holes
from skimage.measure import label, find_contours
from skimage.color import label2rgb
from skimage.segmentation import watershed
from tifffile import imsave

In [ ]:
seg_output_path = os.path.join(output_path, 'spin_mask')
os.makedirs(seg_output_path, exist_ok=True)

for current_sample in tqdm(sorted(adata.obs['sample_id'].unique())):
    print(current_sample)
    current_output_path = os.path.join(seg_output_path, current_sample)
    os.makedirs(current_output_path, exist_ok=True)

    current_obs = adata.obs[adata.obs['sample_id'] == current_sample].copy()

    scaling_factor = 0.25
    img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))
    img[(current_obs['global_y'] * scaling_factor).astype(int).values, (current_obs['global_x'] * scaling_factor).astype(int).values] = current_obs['region_codes'].values

    # Create binary mask
    img = expand_labels(img, distance=30)
    bw_img = img > 0
    img_uint8 = img_as_ubyte(bw_img)
    img_gaussian = gaussian(img_uint8, sigma=3)
    bw_img = img_gaussian > 0
    bg_img = binary_fill_holes(bw_img)

    # Create seed image
    seed_img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))

    for current_region in tqdm(sorted(current_obs['region_codes'].unique())):
        region_obs = current_obs.loc[current_obs['region_codes'] == current_region, :]

        region_img = np.zeros((int(current_obs['global_y'].max() * scaling_factor) + 1, int(current_obs['global_x'].max() * scaling_factor) + 1))
        region_img[(region_obs['global_y'] * scaling_factor).astype(int).values, (region_obs['global_x'] * scaling_factor).astype(int).values] = 1

        bw_img = region_img > 0
        img_uint8 = img_as_ubyte(bw_img)
        img_gaussian = gaussian(img_uint8, sigma=2)
        bw_img = img_gaussian > 0
        bw_img = binary_fill_holes(bw_img)

        objects = label(bw_img)
        seed_img[objects > 0] = current_region

        current_obj_preview = label2rgb(objects, bg_label=0)
        fig, ax = plt.subplots()
        plt.imshow(current_obj_preview)
        plt.savefig(os.path.join(current_output_path, f'{current_region}.png'))
        plt.clf()
        plt.close()
    
    labels = watershed(bg_img, markers=seed_img.astype(int), mask=bg_img)
    fig, ax = plt.subplots()
    plt.imshow(labels)
    plt.savefig(os.path.join(current_output_path, f'mask.png'))
    plt.clf()
    plt.close()
    
    label_bnd = find_boundaries(labels)
    contours = find_contours(label_bnd, level=.8)
    blank_img = np.zeros((int(current_obs['global_x'].max() * scaling_factor) + 1, int(current_obs['global_y'].max() * scaling_factor) + 1))

    figsize=((int(current_obs['global_x'].max() * scaling_factor) + 1) / 1000, (int(current_obs['global_y'].max() * scaling_factor) + 1) / 1000)
    fig, ax = plt.subplots(figsize=figsize)
    plt.imshow(blank_img, cmap='Greys')
    sns.scatterplot(x='global_x_scaled', y='global_y_scaled', hue='region_codes', data=current_obs, 
                    palette='tab10', s=3, alpha=1, legend=False, edgecolor=None)
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], linewidth=.7, color='#5c5c5c')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(current_output_path, 'bnd.png'))
    plt.clf()
    plt.close()

    imsave(os.path.join(current_output_path, f'mask.tif'), labels.astype(np.uint16))
    imsave(os.path.join(current_output_path, f'mask_rgb.tif'), label2rgb(labels, bg_label=0).astype(np.uint8))
    

## Finalize SPIN label

In [ ]:
adata_pfc = sc.read(os.path.join(base_path, "expr/tissue region identification", '2025-03-24-pfc-rgn-label.h5ad'))
adata_pfc

In [ ]:
# Get colormap
region_pl = sns.color_palette("tab10", adata_pfc.obs['region_codes'].nunique())
sns.palplot(region_pl)

sc.pl.embedding(adata_pfc, basis='X_umap_spin', color='coronal_position', palette='tab20')
sc.pl.embedding(adata_pfc, basis='X_umap_spin', color='region_codes', legend_loc='on data', palette=region_pl)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_pfc.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_pfc[adata_pfc.obs['sample_id']==sample], color='region_codes', palette=region_pl,
                  spot_size=200, legend_loc=None, show=False, title=f'{sample}', ax=axes_flat[i])

In [ ]:
import seaborn as sns

current_sample = 'sample24'
sns.scatterplot(data=adata_pfc.obs[adata_pfc.obs['sample_id'] == current_sample], x='global_x_scaled', y='global_y_scaled', hue='region_codes', palette=region_pl, s=10)
# same aspect ratio
plt.gca().set_aspect('equal', adjustable='box')
# remove legend
# plt.legend().remove()


In [ ]:
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample1') & (adata_pfc.obs['global_x_scaled'] > 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample2') & (adata_pfc.obs['global_x_scaled'] > 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample3') & (adata_pfc.obs['global_x_scaled'] < 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample4') & (adata_pfc.obs['global_x_scaled'] < 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'

adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample21') & (adata_pfc.obs['global_x_scaled'] < 2000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample22') & (adata_pfc.obs['global_x_scaled'] < 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample23') & (adata_pfc.obs['global_x_scaled'] < 3000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'
adata_pfc.obs.loc[(adata_pfc.obs['sample_id'] == 'sample24') & (adata_pfc.obs['global_x_scaled'] > 4000) & (adata_pfc.obs['region_codes'] == '5'), 'region_codes'] = '7'

In [ ]:
# sample 1 x > 3000
# sample 2 x > 3000
# sample 3 x < 3000
# sample 4 x < 3000
# sample 21 x < 2000 as brown
# sample 22 x < 3000
# sample 23 x < 3000
# sample 24 x > 4000

In [ ]:
sc.pl.embedding(adata_pfc, basis='X_umap_spin', color='region_codes', legend_loc='on data', palette=region_pl)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_pfc.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_pfc[adata_pfc.obs['sample_id']==sample], color='region_codes', palette=region_pl,
                  spot_size=200, legend_loc=None, show=False, title=f'{sample}', ax=axes_flat[i])

### Assign region labels

#### PFC

In [ ]:
# region labels
transfer_dict_pfc = {
'1': 'MNG',
'2': 'CTX_L6',
'3': 'CTXsp',
'4': 'CTX_L2/3',
'5': 'CTX_mPFC5',
'6': 'CTX_L4',
'7': 'CTX_L5',
'8': 'CTX_PIR',
'9': 'CTX_ILA2/3',
'10': 'FT',
}

In [ ]:
adata_pfc.obs['region_label'] = adata_pfc.obs['region_codes'].map(transfer_dict_pfc)

In [ ]:
adata_pfc.write_h5ad(os.path.join(base_path, "expr/tissue region identification", '2025-04-10-pfc-rgn-label.h5ad'))

#### ST

In [ ]:
adata_st = sc.read(os.path.join(base_path, "expr/tissue region identification", '2025-03-24-st-rgn-label.h5ad'))
adata_st

In [ ]:
# Get colormap
region_pl = sns.color_palette("tab20", adata_st.obs['region_codes'].nunique())
sns.palplot(region_pl)

In [ ]:
sc.pl.embedding(adata_st, basis='X_umap_spin', color='region_codes', legend_loc='on data', palette=region_pl)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_st.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_st[adata_st.obs['sample_id']==sample], color='region_codes', palette=region_pl,
                  spot_size=200, legend_loc=None, show=False, title=f'{sample}', ax=axes_flat[i])

In [ ]:
current_sample = 'sample17'
sns.scatterplot(data=adata_st.obs[adata_st.obs['sample_id'] == current_sample], x='global_x_scaled', y='global_y_scaled', hue='region_codes', palette=region_pl, s=10)
# same aspect ratio
plt.gca().set_aspect('equal', adjustable='box')
# remove legend
# plt.legend().remove()

In [ ]:
# region labels
transfer_dict_st = {
'1': 'STR',
'2': 'STR',
'3': 'PAL',
'4': 'MNG',
'5': 'CTX_L6',
'6': 'CTXsp',
'7': 'CTX_L2/3',
'8': 'CTX_L4',
'9': 'CTX_L5',
'10': 'CTX_AI2/3',
'11': 'FT',
'12': 'VS',
}

In [ ]:
adata_st.obs['region_label'] = adata_st.obs['region_codes'].map(transfer_dict_st)

In [ ]:
adata_st.write_h5ad(os.path.join(base_path, "expr/tissue region identification", '2025-04-10-st-rgn-label.h5ad'))

#### HP

In [ ]:
adata_hp = sc.read(os.path.join(base_path, "expr/tissue region identification", '2025-03-24-hp-rgn-label.h5ad'))
adata_hp

In [ ]:
# Get colormap
region_pl = sns.color_palette("tab20", adata_hp.obs['region_codes'].nunique())
sns.palplot(region_pl)

In [ ]:
sc.pl.embedding(adata_hp, basis='X_umap_spin', color='region_codes', legend_loc='on data', palette=region_pl)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_hp.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_hp[adata_hp.obs['sample_id']==sample], color='region_codes', palette=region_pl,
                  spot_size=200, legend_loc=None, show=False, title=f'{sample}', ax=axes_flat[i])

In [ ]:
current_sample = 'sample13'
sns.scatterplot(data=adata_hp.obs[adata_hp.obs['sample_id'] == current_sample], x='global_x_scaled', y='global_y_scaled', hue='region_codes', palette=region_pl, s=10)
# same aspect ratio
plt.gca().set_aspect('equal', adjustable='box')
# remove legend
# plt.legend().remove()

In [ ]:
# region labels
transfer_dict_hp = {
'1': 'CTX_HIP',
'2': 'CTX_L6',
'3': 'CTX_L6',
'4': 'CTX_DG',
'5': 'CTX_CA1',
'6': 'CTX_CA3',
'7': 'FT',
'8': 'HY',
'9': 'TH_1',
'10': 'TH_2',
'11': 'TH_EPI',
'12': 'TH_RT',
'13': 'VS',
}

In [ ]:
adata_hp.obs['region_label'] = adata_hp.obs['region_codes'].map(transfer_dict_hp)

In [ ]:
# TODO: rerun 

adata_hp.write_h5ad(os.path.join(base_path, "expr/tissue region identification", '2025-04-10-hp-rgn-label.h5ad'))

## Transfer region label

In [ ]:
# adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster.h5ad'))
# adata = sc.read_h5ad(os.path.join(input_path, '2025-02-27-all-sample-cell-typing-lv3-subcluster-rgn-label.h5ad'))
# adata = sc.read_h5ad(os.path.join(input_path, '2025-04-08-finalized-celltyping.h5ad'))
adata = sc.read_h5ad(os.path.join(base_path, '2025-04-11-finalized-ct-rgn.h5ad'))

In [ ]:
adata_pfc = sc.read_h5ad(os.path.join(base_path, 'tissue region identification','2025-04-10-pfc-rgn-label.h5ad'))
adata_st =  sc.read_h5ad(os.path.join(base_path, 'tissue region identification','2025-04-10-st-rgn-label.h5ad'))
adata_hp =  sc.read_h5ad(os.path.join(base_path, 'tissue region identification','2025-04-10-hp-rgn-label.h5ad'))

In [ ]:
region_labels = pd.concat([
    adata_pfc.obs[['region_label']],
    adata_st.obs[['region_label']],
    adata_hp.obs[['region_label']]
])

# Now make sure the indices match
region_labels = region_labels.reindex(adata.obs.index)

# Assign the merged 'region_label' to adata
adata.obs['region_label'] = region_labels['region_label']


In [ ]:
rgn_order = ['CTX_L2/3',
             'CTX_L4',
             'CTX_L5',
             'CTX_L6',
             'CTX_ILA2/3',
             'CTX_mPFC5',
             'CTXsp',
             'CTX_AI2/3',
             'CTX_PIR',
             'CTX_CA1',
             'CTX_CA3',
             'CTX_DG',
             'CTX_HIP',
             'STR',
             'PAL',
             'TH_1',
             'TH_2',
             'TH_EPI',
             'TH_RT',
             'HY',
             'FT',
             'VS',
             'MNG'
             ]


adata.obs['region_label'] = adata.obs['region_label'].cat.reorder_categories(rgn_order)

adata.uns['region_label_order'] = rgn_order


In [ ]:
region_label_color_dict = { 
    'CTX_L2/3': '#AEFF32',
    'CTX_L4': '#57E354',
    'CTX_L5': '#2AB529',
    'CTX_L6': '#486E21',
    'CTX_ILA2/3': '#FFEEB2',
    'CTX_mPFC5': '#F6C34C',
    'CTXsp':'#80FBD2',
    'CTX_AI2/3':'#a8e1eb',
    'CTX_PIR': '#cccccc',
    'CTX_CA1': '#8B09BE',
    'CTX_CA3': '#CB1587',
    'CTX_DG': '#A07CB3',
    'CTX_HIP': '#F7BDC5',
    'STR': '#00C1FD',
    'PAL': '#1f76b3',
    'TH_1': '#f78a88',
    'TH_2': '#EB9FA9',
    'TH_EPI': '#FFC6E5',
    'TH_RT': '#C85C0C',
    'HY': '#ed5e5b',
    'FT': '#101190',
    'VS': '#789AB3',
    'MNG': '#B1D3C3'
}
adata.uns['region_label_color_dict'] = region_label_color_dict

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata.write_h5ad(f"{base_path}/{date}-finalized-ct-rgn.h5ad")